<a href="https://colab.research.google.com/github/Rini43/NLP_Project_Receipe_Reviews_and_Feedback/blob/main/notebooks/NLP_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Libraries

In [90]:
# Data Manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Text Processing
import string
import re
import html

# PyTorch (Required for BERT)
# import torch

# NLTK Libraries
import nltk
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

# NLTK Preprocessing Tools
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Train-Test Split
from sklearn.model_selection import train_test_split

# Text Vectorization (Bag of Words & TF-IDF)
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Classical Machine Learning Models
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

# Model Evaluation Metrics
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report)

# BERT Tokenizer and Pre-trained Model
# from transformers import AutoTokenizer
# from transformers import AutoModel

# Deep learning models
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping

# Used to save and load trained machine learning models
import joblib
import os

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


# Read the Data

In [26]:
# URL of the dataset stored in the GitHub repository
filepath = "https://raw.githubusercontent.com/Rini43/NLP_Project_Receipe_Reviews_and_Feedback/main/data/Recipe%20Reviews%20and%20User%20Feedback%20Dataset.csv"

# Load the dataset
df_review = pd.read_csv(filepath)

# Display the first 5 rows
df_review.head()

,Unnamed: 0,recipe_number,recipe_code,recipe_name,comment_id,user_id,user_name,user_reputation,created_at,reply_count,thumbs_up,thumbs_down,stars,best_score,text
0,0,1,14299,Creamy White Chili,sp_aUSaElGf_14299_c_2G3aneMRgRMZwXqIHmSdXSG1hEM,u_9iFLIhMa8QaG,Jeri326,1,1665619889,0,0,0,5,527,"I tweaked it a little, removed onions because ..."
1,1,1,14299,Creamy White Chili,sp_aUSaElGf_14299_c_2FsPC83HtzCsQAtOxlbL6RcaPbY,u_Lu6p25tmE77j,Mark467,50,1665277687,0,7,0,5,724,Bush used to have a white chili bean and it ma...
2,2,1,14299,Creamy White Chili,sp_aUSaElGf_14299_c_2FPrSGyTv7PQkZq37j92r9mYGkP,u_s0LwgpZ8Jsqq,Barbara566,10,1664404557,0,3,0,5,710,I have a very complicated white chicken chili ...
3,3,1,14299,Creamy White Chili,sp_aUSaElGf_14299_c_2DzdSIgV9qNiuBaLoZ7JQaartoC,u_fqrybAdYjgjG,jeansch123,1,1661787808,2,2,0,0,581,"In your introduction, you mentioned cream chee..."
4,4,1,14299,Creamy White Chili,sp_aUSaElGf_14299_c_2DtZJuRQYeTFwXBoZRfRhBPEXjI,u_XXWKwVhKZD69,camper77,10,1664913823,1,7,0,0,820,Wonderful! I made this for a &#34;Chili/Stew&#...


# EDA

In [27]:
# Identifying numerical and categorical data

df_review.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18182 entries, 0 to 18181
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Unnamed: 0       18182 non-null  int64 
 1   recipe_number    18182 non-null  int64 
 2   recipe_code      18182 non-null  int64 
 3   recipe_name      18182 non-null  object
 4   comment_id       18182 non-null  object
 5   user_id          18182 non-null  object
 6   user_name        18182 non-null  object
 7   user_reputation  18182 non-null  int64 
 8   created_at       18182 non-null  int64 
 9   reply_count      18182 non-null  int64 
 10  thumbs_up        18182 non-null  int64 
 11  thumbs_down      18182 non-null  int64 
 12  stars            18182 non-null  int64 
 13  best_score       18182 non-null  int64 
 14  text             18180 non-null  object
dtypes: int64(10), object(5)
memory usage: 2.1+ MB


In [28]:
# Identifying dataset size

df_review.shape

(18182, 15)

In [29]:
# Statistical Summary

df_review.describe()

,Unnamed: 0,recipe_number,recipe_code,user_reputation,created_at,reply_count,thumbs_up,thumbs_down,stars,best_score
count,18182.000000,18182.000000,18182.000000,18182.000000,1.818200e+04,18182.000000,18182.000000,18182.000000,18182.000000,18182.000000
mean,121.465295,38.689363,21773.667253,2.159608,1.623710e+09,0.014630,1.089264,0.549335,4.288802,153.162138
std,116.747893,29.786647,23965.109637,10.014666,5.468697e+06,0.137974,4.201004,3.470124,1.544786,141.075316
min,0.000000,1.000000,386.000000,0.000000,1.613035e+09,0.000000,0.000000,0.000000,0.000000,0.000000
25%,45.000000,12.000000,6086.000000,1.000000,1.622717e+09,0.000000,0.000000,0.000000,5.000000,100.000000
50%,91.000000,33.000000,14600.000000,1.000000,1.622718e+09,0.000000,0.000000,0.000000,5.000000,100.000000
75%,150.000000,64.000000,33121.000000,1.000000,1.622718e+09,0.000000,0.000000,0.000000,5.000000,100.000000
max,724.000000,100.000000,191775.000000,520.000000,1.665756e+09,3.000000,106.000000,126.000000,5.000000,946.000000


In [30]:
# Target Variable Analysis

df_review["stars"].value_counts()

# 1–5 are actual rating classes.
# 0 is not a rating, it means the user did not provide a rating.

,count
stars,
5,13829
0,1696
4,1655
3,490
1,280
2,232


In [31]:
# Trying to findout if there is any missing value present

df_review.isna().sum()

,0
Unnamed: 0,0
recipe_number,0
recipe_code,0
recipe_name,0
comment_id,0
user_id,0
user_name,0
user_reputation,0
created_at,0
reply_count,0


In [32]:
# Handling the missing value

df_review.dropna(subset=["text"], inplace=True)

In [33]:
# After handling the misssing values

df_review.isna().sum()

,0
Unnamed: 0,0
recipe_number,0
recipe_code,0
recipe_name,0
comment_id,0
user_id,0
user_name,0
user_reputation,0
created_at,0
reply_count,0


In [34]:
# Duplicate Rows

df_review.duplicated().sum()

np.int64(0)

In [35]:
# Count the number of unique reviews in the 'text' column

df_review["text"].nunique()

17731

In [36]:
# Duplicate review text

df_review["text"].duplicated().sum()

np.int64(449)

In [37]:
#  Inspect them

df_review[df_review["text"].duplicated(keep=False)].sort_values("text")

,Unnamed: 0,recipe_number,recipe_code,recipe_name,comment_id,user_id,user_name,user_reputation,created_at,reply_count,thumbs_up,thumbs_down,stars,best_score,text
252,252,1,14299,Creamy White Chili,sp_aUSaElGf_14299_c_107272,u_1oKVaAf9Tks6DDSWzXwN6Q5WKBN,ziki01,1,1622716893,0,0,0,5,100,.
6016,48,17,36450,Fluffy Key Lime Pie,sp_aUSaElGf_36450_c_107441,u_1oKVZb1mFfcT6ORS5FagojvK7rH,GStaelens,1,1622716887,0,0,1,3,100,.
167,167,1,14299,Creamy White Chili,sp_aUSaElGf_14299_c_107293,u_1oKVaKbaDb9XpuJSTzLKD4ult94,taylor,1,1622716893,0,0,0,5,100,.
4284,104,11,12003,Traditional Lasagna,sp_aUSaElGf_12003_c_107332,u_1oKVZb1mFfcT6ORS5FagojvK7rH,GStaelens,1,1622716881,0,0,0,5,100,.
16064,88,82,18274,Ravioli Lasagna,sp_aUSaElGf_18274_c_107642,u_1oKVZpBH6PLJXY2WdzR0I1YxFlr,smvlpn,1,1622716887,0,0,0,4,100,.
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6332,138,18,2872,Stuffed Pepper Soup,sp_aUSaElGf_2872_c_378868,u_1oKb4IFxLy8R9TkSJiPthlA5aj1,Lisabarney671,1,1622718230,0,0,0,5,100,yummy
10050,150,38,1063,Frosted Banana Bars,sp_aUSaElGf_1063_c_106240,u_1oKVZlbJsKnks4yML21aFN0AsFC,saw-whet,1,1622716880,0,0,0,0,100,
15049,122,74,26937,Pineapple Pudding Cake,sp_aUSaElGf_26937_c_106837,u_1oKVeA3xzAOfmCXeDrITZ2mGGIR,conshanty,1,1622716881,0,0,0,5,100,
5693,247,15,10252,Li’l Cheddar Meat Loaves,sp_aUSaElGf_10252_c_107176,u_1oKVZeTXJSMyvTZ0ijow6MBp0zM,ScottsdalePrincess,1,1622716888,0,0,0,0,100,My son loves these. I have made them many man...


In [38]:
# Remove duplicate reviews based on the 'text' column
# Keep only the first occurrence of each review
# Reset the index after removing duplicates

df_review = df_review.drop_duplicates(
    subset=["text"],
    keep="first"
).reset_index(drop=True)

In [39]:
# Checking the review text

df_review["text"].duplicated().sum()

np.int64(0)

In [40]:
# removing unwanted colums
df_review.drop(columns = ['Unnamed: 0','recipe_number','recipe_code','comment_id','user_id','user_reputation','created_at','reply_count','best_score'],inplace = True)

In [41]:
df_review.drop(columns = ['thumbs_up','thumbs_down','user_name'],inplace = True)

# NLP Preprocessing

## Lowercasing

In [42]:
# Convert to lowercase

df_review['recipe_name'] = df_review['recipe_name'].str.lower()
df_review['text'] = df_review['text'].astype(str).str.lower()

## Punctuation

In [43]:
# Remove punctuation

text_columns = ["recipe_name", "text"]

for col in text_columns:
    df_review[col] = df_review[col].fillna("").str.lower()
    df_review[col] = df_review[col].str.translate(
        str.maketrans("", "", string.punctuation))

## URL

In [44]:
# Checking for the URLs

url_count = df_review["text"].str.contains(r"http\S+|www\.\S+", regex=True, na=False).sum()
print(f"Reviews containing URLs in text: {url_count}")

url_count = df_review["recipe_name"].str.contains(r"http\S+|www\.\S+", regex=True, na=False).sum()
print(f"Reviews containing URLs in recipe name: {url_count}")

Reviews containing URLs in text: 34
Reviews containing URLs in recipe name: 0


In [45]:
# Remove URLs

df_review["text"] = df_review["text"].str.replace(r"http\S+|www\.\S+", "", regex=True)

## HTML

In [46]:
# Convert HTML entities to normal characters and ensure text is in string format

df_review["text"] = df_review["text"].apply(
    lambda x: html.unescape(str(x))
)

# Emoji

In [47]:
# Checking for emojis

# Compile emoji pattern once

emoji_pattern = re.compile(
    "["
    "\U0001F600-\U0001F64F"
    "\U0001F300-\U0001F5FF"
    "\U0001F680-\U0001F6FF"
    "\U0001F1E0-\U0001F1FF"
    "]+",
    flags=re.UNICODE)

# Check multiple columns
text_columns = ["text", "recipe_name"]

for col in text_columns:
    emoji_count = df_review[col].apply(
        lambda x: bool(emoji_pattern.search(str(x)))).sum()

    print(f"Rows containing emojis in '{col}': {emoji_count}")

Rows containing emojis in 'text': 11
Rows containing emojis in 'recipe_name': 0


In [48]:
# Removing the emojis

df_review["text"] = df_review["text"].apply(
    lambda x: emoji_pattern.sub("", str(x)))

## Numbers

In [49]:
# Checking for numbers

text_columns = ["text", "recipe_name"]

for col in text_columns:
    number_count = df_review[col].str.contains(r"\d", regex=True, na=False).sum()
    print(f"Rows containing numbers in '{col}': {number_count}")

Rows containing numbers in 'text': 8776
Rows containing numbers in 'recipe_name': 0


In [50]:
# Display rows containing numbers

number_rows = df_review[df_review["text"].str.contains(r"\d", regex=True, na=False)]
number_rows["text"].head(20)

,text
4,wonderful i made this for a 34chilistew34 nigh...
6,wow this recipe is excellent as written the ...
7,this is delicious and i make it often one such...
8,i absolutely love this recipe i39ve tweaked it...
11,best white chili recipe i39ve had i served it ...
12,this recipe was excellent i added the cream ch...
14,fantastic but mild i added half a carolina rea...
21,this is our goto chicken chili recipe and has ...
23,this is just white chicken chili with i first ...
24,wow total wow totally delicious 5 stars plus


In [51]:
# Convert the 'text' column to string data type
df_review["text"] = df_review["text"].astype("string")

# Check the data type of the 'text' column
print(df_review["text"].dtype)

string


In [52]:
# Convert star ratings into sentiment labels
# 1–2 stars → Negative sentiment
# 3 stars   → Neutral sentiment
# 4–5 stars → Positive sentiment

def create_sentiment(stars):
    if stars in [1, 2]:
        return "Negative"
    elif stars == 3:
        return "Neutral"
    elif stars in [4, 5]:
        return "Positive"
    return None


# Create a new 'sentiment' column based on the star ratings
df_review["sentiment"] = df_review["stars"].apply(create_sentiment)

# Remove reviews with no valid sentiment label, such as unrated reviews (0 stars)
df_review = df_review.dropna(subset=["sentiment"]).copy()

In [53]:
# Count the number of reviews in each sentiment category
# This helps check the distribution of Negative, Neutral, and Positive reviews

df_review["sentiment"].value_counts()

,count
sentiment,
Positive,15107
Negative,509
Neutral,476


## Tokenization

In [54]:
text = df_review['text'].apply(word_tokenize)

## Stopword Removal

In [55]:
stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
  tokens = word_tokenize(text)
  filtered = [word for word in tokens if word not in stop_words]
  return " ".join(filtered)

df_review['text'] = df_review['text'].apply(remove_stopwords)

## Lemmatization

In [56]:
lemmatizer = WordNetLemmatizer()

def lemmatize_text(text):
  tokens = word_tokenize(text)
  lemmas = [lemmatizer.lemmatize(word) for word in tokens]
  return " ".join(lemmas)

df_review['text'] = df_review['text'].apply(lemmatize_text)

# Classical Machine Learning Models

In [57]:
# Define Features and Target

X = df_review["text"]

# Target variable
y = df_review["stars"]

### Train-Test Split

In [58]:
# Train-Test Split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42,
    stratify=y)

# The vectorizer is fitted only on the training data to prevent data leakage.
# The learned vocabulary is then used to transform both the training and test data.

In [59]:
# Verify the shapes

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (12873,)
X_test : (3219,)
y_train: (12873,)
y_test : (3219,)


# Text Vectorization

## Bag of Words

In [60]:
# Bag of Words

bow = CountVectorizer(
    max_features=5000,
    ngram_range=(1,2),
    min_df=2,
    max_df=0.95)

X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)

# we are checking for the word counts

In [61]:
# Checking the Output

print("Training Shape :", X_train_bow.shape)
print("Testing Shape  :", X_test_bow.shape)

print("Vocabulary Size :", len(bow.vocabulary_))

Training Shape : (12873, 5000)
Testing Shape  : (3219, 5000)
Vocabulary Size : 5000


In [62]:
# View Vocabulary

feature_names = bow.get_feature_names_out()
print(feature_names[:50])

['10' '10 min' '10 minute' '10 star' '10 year' '100' '11' '112' '112 cup'
 '12' '12 cup' '12 hour' '12 lb' '12 oz' '12 pound' '12 recipe'
 '12 teaspoon' '12 tsp' '12 year' '125' '12c' '13' '13 cup' '13 pan'
 '13x9' '14' '14 cup' '14 oz' '14 teaspoon' '14 tsp' '15' '15 min'
 '15 minute' '15 year' '15x10' '16' '16 oz' '18' '19' '1996' '1lb' '1st'
 '1st time' '1tsp' '20' '20 min' '20 minute' '20 year' '23' '23 cup']


In [63]:
# Evaluation Function

def evaluate_model(model, X_test, y_test):

    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)

    precision = precision_score(y_test, y_pred,
        average="weighted")

    recall = recall_score(y_test, y_pred,
        average="weighted")

    f1 = f1_score(y_test, y_pred,
        average="weighted")

    print("Accuracy :", round(accuracy,4))
    print("Precision:", round(precision,4))
    print("Recall   :", round(recall,4))
    print("F1 Score :", round(f1,4))

    print("\nClassification Report\n")
    print(classification_report(y_test,y_pred))

    print("\nConfusion Matrix\n")
    print(confusion_matrix(y_test,y_pred))

    return accuracy, precision, recall, f1

In [64]:
# Creating Results List

results = []

In [65]:
# Train Logistic Regression

lr = LogisticRegression(
    max_iter=1000,
    random_state=42)

lr.fit(X_train_bow, y_train)

accuracy, precision, recall, f1 = evaluate_model(
    lr, X_test_bow, y_test)

results.append({
    "Embedding":"Bag of Words",
    "Model":"Logistic Regression",
    "Accuracy":accuracy,
    "Precision":precision,
    "Recall":recall,
    "F1 Score":f1})

Accuracy : 0.8326
Precision: 0.7994
Recall   : 0.8326
F1 Score : 0.8127

Classification Report

              precision    recall  f1-score   support

           1       0.54      0.25      0.34        56
           2       0.12      0.07      0.09        46
           3       0.27      0.18      0.22        95
           4       0.33      0.22      0.27       323
           5       0.89      0.95      0.92      2699

    accuracy                           0.83      3219
   macro avg       0.43      0.33      0.37      3219
weighted avg       0.80      0.83      0.81      3219


Confusion Matrix

[[  14    5    7    6   24]
 [   5    3   10   10   18]
 [   2    7   17   22   47]
 [   1    7   17   72  226]
 [   4    2   12  107 2574]]


In [66]:
# Train SVM

svm = SVC(
    kernel="linear",
    probability=True,
    random_state=42)

svm.fit(X_train_bow, y_train)

accuracy, precision, recall, f1 = evaluate_model(
    svm, X_test_bow, y_test)

results.append({
    "Embedding":"Bag of Words",
    "Model":"SVM",
    "Accuracy":accuracy,
    "Precision":precision,
    "Recall":recall,
    "F1 Score":f1})

Accuracy : 0.7999
Precision: 0.7957
Recall   : 0.7999
F1 Score : 0.7975

Classification Report

              precision    recall  f1-score   support

           1       0.31      0.30      0.31        56
           2       0.10      0.13      0.12        46
           3       0.21      0.25      0.23        95
           4       0.27      0.22      0.24       323
           5       0.90      0.91      0.91      2699

    accuracy                           0.80      3219
   macro avg       0.36      0.36      0.36      3219
weighted avg       0.80      0.80      0.80      3219


Confusion Matrix

[[  17    8    6    5   20]
 [  10    6   13    6   11]
 [   4   11   24   19   37]
 [   8   15   26   72  202]
 [  15   18   46  164 2456]]


In [67]:
# Display Final Results

results_df = pd.DataFrame(results)
results_df

,Embedding,Model,Accuracy,Precision,Recall,F1 Score
0,Bag of Words,Logistic Regression,0.832557,0.799449,0.832557,0.812712
1,Bag of Words,SVM,0.799938,0.795687,0.799938,0.797476


## TF-IDF Vectorization

In [68]:
# TF-IDF

tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2),
    min_df=2,
    max_df=0.95)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# we are checking for the weighted word frequencies

In [69]:
# Checking the Output

print("Training Shape :", X_train_tfidf.shape)
print("Testing Shape  :", X_test_tfidf.shape)

print("Vocabulary Size :", len(tfidf.vocabulary_))

Training Shape : (12873, 5000)
Testing Shape  : (3219, 5000)
Vocabulary Size : 5000


In [70]:
# Train Logistic Regression

lr = LogisticRegression(max_iter=1000, random_state=42)

lr.fit(X_train_tfidf, y_train)

accuracy, precision, recall, f1 = evaluate_model(
    lr, X_test_tfidf, y_test)

results.append({
    "Embedding":"TF-IDF",
    "Model":"Logistic Regression",
    "Accuracy":accuracy,
    "Precision":precision,
    "Recall":recall,
    "F1 Score":f1})

Accuracy : 0.8481
Precision: 0.8
Recall   : 0.8481
F1 Score : 0.7995

Classification Report

              precision    recall  f1-score   support

           1       0.75      0.11      0.19        56
           2       0.50      0.02      0.04        46
           3       0.27      0.04      0.07        95
           4       0.50      0.13      0.21       323
           5       0.86      0.99      0.92      2699

    accuracy                           0.85      3219
   macro avg       0.58      0.26      0.29      3219
weighted avg       0.80      0.85      0.80      3219


Confusion Matrix

[[   6    0    3    1   46]
 [   1    1    5    7   32]
 [   1    1    4   12   77]
 [   0    0    3   42  278]
 [   0    0    0   22 2677]]


In [71]:
# Train SVM

svm = SVC(
    kernel="linear",
    probability=True,
    random_state=42)

svm.fit(X_train_tfidf, y_train)

accuracy, precision, recall, f1 = evaluate_model(
    svm, X_test_tfidf, y_test)

results.append({
    "Embedding":"TF-IDF",
    "Model":"SVM",
    "Accuracy":accuracy,
    "Precision":precision,
    "Recall":recall,
    "F1 Score":f1})

Accuracy : 0.8459
Precision: 0.7923
Recall   : 0.8459
F1 Score : 0.7877

Classification Report

              precision    recall  f1-score   support

           1       0.62      0.14      0.23        56
           2       0.25      0.02      0.04        46
           3       0.35      0.07      0.12        95
           4       0.52      0.04      0.08       323
           5       0.85      1.00      0.92      2699

    accuracy                           0.85      3219
   macro avg       0.52      0.26      0.28      3219
weighted avg       0.79      0.85      0.79      3219


Confusion Matrix

[[   8    1    3    0   44]
 [   2    1    6    3   34]
 [   3    1    7    4   80]
 [   0    1    4   14  304]
 [   0    0    0    6 2693]]


In [72]:
# Display Final Results

results_df = pd.DataFrame(results)
results_df

,Embedding,Model,Accuracy,Precision,Recall,F1 Score
0,Bag of Words,Logistic Regression,0.832557,0.799449,0.832557,0.812712
1,Bag of Words,SVM,0.799938,0.795687,0.799938,0.797476
2,TF-IDF,Logistic Regression,0.848089,0.799955,0.848089,0.799498
3,TF-IDF,SVM,0.845915,0.792317,0.845915,0.787654


## BERT Embedding

* Dense contextual embeddings
* It takes much longer to generate embeddings.
* It uses more RAM and computation.

Therefore, in this project, only the BERT model implementation is provided as a reference and is not used for the final model comparison.

In [73]:
# Loading the Pre-trained BERT

# tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
# bert_model = AutoModel.from_pretrained("bert-base-uncased")

In [74]:
# BERT Embedding Function

# def get_bert_embeddings(texts):

#    embeddings = []
#    bert_model.eval()
#    with torch.no_grad():

#       for text in texts:

#            encoded = tokenizer(
#                str(text),
#                padding="max_length",
#                truncation=True,
#                max_length=128,
#                return_tensors="pt")

#            output = bert_model(**encoded)

#            cls_embedding = output.last_hidden_state[:,0,:].squeeze().numpy()
#            embeddings.append(cls_embedding)

#    return np.array(embeddings)

In [75]:
# Generate Embeddings

# X_train_bert = get_bert_embeddings(X_train)
# X_test_bert = get_bert_embeddings(X_test)

In [76]:
# Checking the Shape

# print(X_train_bert.shape)
# print(X_test_bert.shape)

In [77]:
# Train Logistic Regression

# lr = LogisticRegression(
#    max_iter=1000,
#    random_state=42)

# lr.fit(X_train_bert, y_train)

# accuracy, precision, recall, f1 = evaluate_model(
#    lr, X_test_bert, y_test)

# results.append({
#    "Embedding":"BERT",
#    "Model":"Logistic Regression",
#   "Accuracy":accuracy,
#    "Precision":precision,
#    "Recall":recall,
#   "F1 Score":f1})

In [78]:
# Train SVM

# svm = SVC(
#    kernel="linear",
#    probability=True,
#    random_state=42)

# svm.fit(X_train_bert, y_train)

# accuracy, precision, recall, f1 = evaluate_model(
#    svm, X_test_bert, y_test)

# results.append({
#    "Embedding":"BERT",
#    "Model":"SVM",
#    "Accuracy":accuracy,
#    "Precision":precision,
#    "Recall":recall,
#    "F1 Score":f1})

In [79]:
# Display Final Results

# results_df = pd.DataFrame(results)
# results_df

# Deep Learning Model

### Tokenization

In [80]:
# Convert review text into numerical sequences

# Maximum number of words to keep in the vocabulary
MAX_WORDS = 10000

# Maximum number of tokens allowed in each review
MAX_LENGTH = 200

# Create a tokenizer and replace unknown words with <OOV>
tokenizer = Tokenizer(
    num_words=MAX_WORDS,
    oov_token="<OOV>"
)

# Learn the vocabulary only from the training reviews
tokenizer.fit_on_texts(X_train)

# Convert training reviews into numerical sequences
X_train_seq = tokenizer.texts_to_sequences(X_train)

# Convert test reviews into numerical sequences
# using the vocabulary learned from the training data
X_test_seq = tokenizer.texts_to_sequences(X_test)

# Pad training sequences to a fixed length
X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=MAX_LENGTH,
    padding="post",
    truncating="post"
)

# Pad test sequences to the same fixed length
X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=MAX_LENGTH,
    padding="post",
    truncating="post"
)

### Building BiLSTM Model

In [81]:
# Create Sequential model

model = Sequential()

# Word Embedding Layer
# Converts words into dense vector representations

model.add(Embedding(input_dim=MAX_WORDS, output_dim=128, input_length=MAX_LENGTH))

# Bidirectional LSTM Layer
# Reads the text in both forward and backward directions

model.add(Bidirectional(LSTM(64)))

# Dropout Layer
# Reduces overfitting

model.add(Dropout(0.5))

# Hidden Dense Layer

model.add(Dense(64, activation='relu'))

# Dropout

model.add(Dropout(0.3))

# Output Layer
# Number of neurons = Number of classes (5)

model.add(Dense(6, activation='softmax'))     # Multiclass

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [82]:
# Compile the model

model.compile(optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'])

# Display model summary

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

### Training the Model

In [83]:
# Early stopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True)

# Train the model

history = model.fit(
    X_train_pad,
    y_train,
    epochs=5,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop])

Epoch 1/5
322/322 ━━━━━━━━━━━━━━━━━━━━ 14s 20ms/step - accuracy: 0.8362 - loss: 0.6142 - val_accuracy: 0.8392 - val_loss: 0.4793
Epoch 2/5
322/322 ━━━━━━━━━━━━━━━━━━━━ 7s 21ms/step - accuracy: 0.8498 - loss: 0.4429 - val_accuracy: 0.8478 - val_loss: 0.4599
Epoch 3/5
322/322 ━━━━━━━━━━━━━━━━━━━━ 6s 19ms/step - accuracy: 0.8688 - loss: 0.3736 - val_accuracy: 0.8462 - val_loss: 0.4873
Epoch 4/5
322/322 ━━━━━━━━━━━━━━━━━━━━ 7s 21ms/step - accuracy: 0.8921 - loss: 0.3050 - val_accuracy: 0.8256 - val_loss: 0.5243
Epoch 5/5
322/322 ━━━━━━━━━━━━━━━━━━━━ 6s 19ms/step - accuracy: 0.9093 - loss: 0.2545 - val_accuracy: 0.8159 - val_loss: 0.5856


In [84]:
 # Evaluate the model

loss, accuracy = model.evaluate(X_test_pad, y_test)

print("Test Loss :", loss)
print("Test Accuracy :", accuracy) # Predict star ratings

y_pred = model.predict(X_test_pad)

101/101 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.8434 - loss: 0.4690
Test Loss : 0.46896812319755554
Test Accuracy : 0.8434296250343323
101/101 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step


### Predictions of the BiLSTM model

In [85]:
# Convert probabilities into class labels

y_pred = np.argmax(y_pred, axis=1)  # Calculate evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


### Evaluation

In [86]:
# Append LSTM results to the results list

results.append({
    "Embedding": "Tokenizer + Embedding",
    "Model": "LSTM",
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1 Score": f1})


### Classification Report

In [87]:
# Classification Report
print(classification_report(y_test, y_pred))
# Confusion Matrix
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           1       0.40      0.14      0.21        56
           2       0.00      0.00      0.00        46
           3       0.26      0.13      0.17        95
           4       0.33      0.20      0.25       323
           5       0.89      0.97      0.93      2699

    accuracy                           0.84      3219
   macro avg       0.38      0.29      0.31      3219
weighted avg       0.79      0.84      0.81      3219

[[   8    0    4   25   19]
 [   5    0   10   15   16]
 [   5    0   12   29   49]
 [   2    0   12   65  244]
 [   0    0    8   61 2630]]


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# Save the Model

In [91]:
# Create a folder named 'models' to store trained model files
os.makedirs("models", exist_ok=True)

# Save the trained TF-IDF vectorizer for future text preprocessing
joblib.dump(tfidf, "models/tfidf_vectorizer.pkl")

# Save the trained Logistic Regression model for sentiment prediction
joblib.dump(lr, "models/recipe_rating_model.pkl")

['models/recipe_rating_model.pkl']